# 07 · Paper assets

Turns the CSV results into the LaTeX tables included by the paper, and copies the figures
into `paper/figures`, so that every number in the PDF comes from a file written by notebooks
01-06 rather than being typed by hand.

In [1]:
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src import config as C

PAPER = C.ROOT / "paper"
TAB = PAPER / "eval"
TAB.mkdir(parents=True, exist_ok=True)
(PAPER / "figures").mkdir(exist_ok=True)
for f in C.FIGURES.glob("fig*.pdf"):
    shutil.copy(f, PAPER / "figures" / f.name)
for f in C.FIGURES.glob("fig*.png"):
    shutil.copy(f, PAPER / "figures" / f.name)
print(len(list((PAPER / "figures").glob("fig*.pdf"))), "figures copied")


def pm(mu, sd, nd=3):
    return f"{mu:.{nd}f} $\\pm$ {sd:.{nd}f}"


def bold_best(values, higher=True):
    arr = np.array(values, dtype=float)
    best = np.nanmax(arr) if higher else np.nanmin(arr)
    return [np.isclose(v, best, atol=5e-4) for v in arr]


def write(name, body):
    # shrink wide tables to the text width, never enlarge narrow ones
    body = body.replace(r"\begin{tabular}", r"\begin{adjustbox}{max width=\textwidth}" + "\n" + r"\begin{tabular}")
    body = body.replace(r"\end{tabular}", r"\end{tabular}" + "\n" + r"\end{adjustbox}")
    (TAB / name).write_text(body)
    print(f"--- {name}\n{body}")

22 figures copied


## EDA statistics

In [2]:
mw = pd.read_csv(C.METRICS / "eda_mannwhitney.csv")
lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{Window descriptors: medians and Mann--Whitney rank-biserial effect size $r$ "
         r"(all $p<10^{-10}$ unless stated).}", r"\label{tab:mannwhitney}",
         r"\begin{tabular}{llrrr}", r"\toprule",
         r"Comparison (A vs B) & Descriptor & Median A & Median B & $r$ \\", r"\midrule"]
prev = None
for _, r in mw.iterrows():
    comp = r.comparison if r.comparison != prev else ""
    prev = r.comparison
    ptxt = "" if r.p < 1e-10 else f" ($p={r.p:.1e}$)"
    lines.append(f"{comp} & {r.descriptor.replace('|a|', '$|a|$')} & {r['median A']:.3f} & "
                 f"{r['median B']:.3f} & {r['effect r (rank-biserial)']:.2f}{ptxt} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_mannwhitney.tex", "\n".join(lines))

--- table_mannwhitney.tex
\begin{table}[ht]
\centering
\small
\caption{Window descriptors: medians and Mann--Whitney rank-biserial effect size $r$ (all $p<10^{-10}$ unless stated).}
\label{tab:mannwhitney}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{llrrr}
\toprule
Comparison (A vs B) & Descriptor & Median A & Median B & $r$ \\
\midrule
SisFall ADL vs SisFall fall & $|a|$ std (g) & 0.055 & 0.714 & 0.82 \\
 & peak $|a|$ (g) & 1.179 & 5.575 & 0.94 \\
 & freeze index (log10) & -0.338 & -0.273 & 0.09 ($p=2.5e-08$) \\
Daphnet walk vs Daphnet FoG & $|a|$ std (g) & 0.047 & 0.104 & 0.37 \\
 & peak $|a|$ (g) & 1.172 & 1.389 & 0.34 \\
 & freeze index (log10) & 0.077 & 0.381 & 0.33 \\
SisFall ADL vs Daphnet walk & $|a|$ std (g) & 0.055 & 0.047 & -0.04 ($p=2.1e-05$) \\
 & peak $|a|$ (g) & 1.179 & 1.172 & 0.01 ($p=5.9e-01$) \\
 & freeze index (log10) & -0.338 & 0.077 & 0.42 \\
\bottomrule
\end{tabular}
\end{adjustbox}
\end{table}


## Selected configurations

In [3]:
best = json.load(open(C.METRICS / "best_hparams.json"))
runs = pd.read_csv(C.METRICS / "final_runs.csv")


def describe(arch, hp):
    if arch == "CNN-1D":
        return f"width {hp['width']}, kernel {hp['kernel']}, dropout {hp['dropout']}"
    if arch in ("LSTM", "GRU"):
        return (f"hidden {hp['hidden']}, {hp['layers']} layer{'s' if hp['layers'] > 1 else ''}, "
                f"{'bidirectional' if hp['bidirectional'] else 'unidirectional'}, dropout {hp['dropout']}")
    return f"$d={hp['d_model']}$, {hp['layers']} layers, patch {hp['patch']}, dropout {hp['dropout']}"


lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{Configurations selected on SisFall validation, trainable parameters, mean epochs "
         r"to the selected checkpoint and training time per epoch on one NVIDIA T4 GPU.}", r"\label{tab:hparams}",
         r"\begin{tabular}{lp{6.3cm}rrrr}", r"\toprule",
         r"Model & Configuration & lr & Params & Best ep. & s/epoch \\", r"\midrule"]
for arch in C.ARCHS:
    hp = best[arch]
    rr = runs[runs.arch == arch]
    lines.append(f"{arch} & {describe(arch, hp)} & {hp['lr']:.0e} & {int(rr.params.iloc[0]):,} & "
                 f"{rr.best_epoch.mean():.1f} & {rr.sec_per_epoch.mean():.0f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_hparams.tex", "\n".join(lines).replace("e-04", "e-4").replace("e-03", "e-3"))

--- table_hparams.tex
\begin{table}[ht]
\centering
\small
\caption{Configurations selected on SisFall validation, trainable parameters, mean epochs to the selected checkpoint and training time per epoch on one NVIDIA T4 GPU.}
\label{tab:hparams}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lp{6.3cm}rrrr}
\toprule
Model & Configuration & lr & Params & Best ep. & s/epoch \\
\midrule
CNN-1D & width 32, kernel 7, dropout 0.5 & 3e-3 & 36,289 & 5.7 & 2 \\
LSTM & hidden 64, 2 layers, bidirectional, dropout 0.3 & 1e-3 & 135,425 & 8.0 & 4 \\
GRU & hidden 32, 2 layers, bidirectional, dropout 0.1 & 3e-3 & 26,241 & 9.7 & 2 \\
Transformer & $d=64$, 2 layers, patch 8, dropout 0.2 & 3e-3 & 70,401 & 7.3 & 3 \\
\bottomrule
\end{tabular}
\end{adjustbox}
\end{table}


## Main results: in-domain and zero-shot

In [4]:
test = pd.read_csv(C.METRICS / "sisfall_test_per_seed.csv")
zs = pd.read_csv(C.METRICS / "daphnet_zero_shot_per_seed.csv")
zs_main = zs[zs.normalisation == "domain statistics"]
COLS = [("auroc", "AUROC", True), ("auprc", "AUPRC", True), ("recall", "Recall", True),
        ("specificity", "Spec.", True), ("f1", "F$_1$", True), ("false_alarms_per_hour", "FA/h", False)]


def result_table(df, caption, label, cols=COLS):
    g = df.groupby("arch")
    mu, sd = g.mean(numeric_only=True).loc[C.ARCHS], g.std(numeric_only=True).loc[C.ARCHS]
    head = " & ".join(["Model"] + [c[1] for c in cols])
    lines = [r"\begin{table}[ht]", r"\centering", r"\footnotesize", f"\\caption{{{caption}}}",
             f"\\label{{{label}}}", r"\setlength{\tabcolsep}{4pt}",
             r"\begin{tabular}{l" + "c" * len(cols) + "}", r"\toprule", head + r" \\", r"\midrule"]
    flags = {c: bold_best(mu[c].values, hi) for c, _, hi in cols}
    for i, arch in enumerate(C.ARCHS):
        cells = []
        for c, _, _ in cols:
            nd = 1 if c == "false_alarms_per_hour" else 3
            txt = pm(mu.loc[arch, c], sd.loc[arch, c], nd)
            cells.append(f"\\textbf{{{txt}}}" if flags[c][i] else txt)
        lines.append(" & ".join([arch] + cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


write("table_sisfall.tex", result_table(
    test, r"In-domain fall detection on the eight SisFall test subjects (mean $\pm$ std over three seeds). "
          r"Recall, specificity, F$_1$ and false alarms per hour (FA/h, one negative window per second of "
          r"activity) use the threshold selected on validation. Best mean in bold.", "tab:sisfall"))
write("table_zeroshot.tex", result_table(
    zs_main, r"Zero-shot transfer to Daphnet (all 17 recordings, independent domain statistics, "
             r"SisFall-validated threshold). Positives are FoG windows. Best mean in bold.", "tab:zeroshot"))

--- table_sisfall.tex
\begin{table}[ht]
\centering
\footnotesize
\caption{In-domain fall detection on the eight SisFall test subjects (mean $\pm$ std over three seeds). Recall, specificity, F$_1$ and false alarms per hour (FA/h, one negative window per second of activity) use the threshold selected on validation. Best mean in bold.}
\label{tab:sisfall}
\setlength{\tabcolsep}{4pt}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Model & AUROC & AUPRC & Recall & Spec. & F$_1$ & FA/h \\
\midrule
CNN-1D & 0.987 $\pm$ 0.003 & 0.980 $\pm$ 0.002 & \textbf{0.980 $\pm$ 0.002} & \textbf{1.000 $\pm$ 0.000} & 0.982 $\pm$ 0.006 & 1.5 $\pm$ 1.1 \\
LSTM & \textbf{0.999 $\pm$ 0.000} & 0.990 $\pm$ 0.002 & 0.977 $\pm$ 0.010 & \textbf{1.000 $\pm$ 0.000} & \textbf{0.985 $\pm$ 0.005} & 0.8 $\pm$ 0.0 \\
GRU & \textbf{0.999 $\pm$ 0.000} & \textbf{0.992 $\pm$ 0.002} & 0.977 $\pm$ 0.003 & \textbf{1.000 $\pm$ 0.000} & \textbf{0.985 $\pm$ 0.002} & \textbf{0.7 $\pm$ 0.1} \\
Transformer & 

In [5]:
ci = pd.read_csv(C.METRICS / "auroc_subject_bootstrap_ci.csv").set_index("arch").loc[C.ARCHS]
lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{AUROC of the seed-averaged scores with 95\% subject-level bootstrap intervals.}",
         r"\label{tab:ci}", r"\begin{tabular}{lcc}", r"\toprule",
         r"Model & SisFall test & Daphnet zero-shot \\", r"\midrule"]
for arch, r in ci.iterrows():
    lines.append(f"{arch} & {r.sisfall_test_auroc_ensemble:.3f} [{r.sisfall_ci_low:.3f}, {r.sisfall_ci_high:.3f}] & "
                 f"{r.daphnet_auroc_ensemble:.3f} [{r.daphnet_ci_low:.3f}, {r.daphnet_ci_high:.3f}] \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_ci.tex", "\n".join(lines))

--- table_ci.tex
\begin{table}[ht]
\centering
\small
\caption{AUROC of the seed-averaged scores with 95\% subject-level bootstrap intervals.}
\label{tab:ci}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcc}
\toprule
Model & SisFall test & Daphnet zero-shot \\
\midrule
CNN-1D & 0.985 [0.961, 1.000] & 0.473 [0.407, 0.546] \\
LSTM & 1.000 [0.999, 1.000] & 0.581 [0.381, 0.711] \\
GRU & 0.999 [0.998, 1.000] & 0.639 [0.458, 0.765] \\
Transformer & 0.999 [0.997, 1.000] & 0.650 [0.548, 0.733] \\
\bottomrule
\end{tabular}
\end{adjustbox}
\end{table}


## Normalisation ablation

In [6]:
VIEWS = ["source statistics", "domain statistics", "subject statistics"]
g = zs.groupby(["normalisation", "arch"])
mu, sd = g.auroc.mean(), g.auroc.std()
rec_mu, fpr_mu = g.recall.mean(), g.fpr.mean()
lines = [r"\begin{table}[ht]", r"\centering", r"\footnotesize",
         r"\caption{Zero-shot Daphnet results under three label-free normalisations of the target "
         r"inputs: AUROC (mean $\pm$ std over seeds) and, in brackets, recall / false-positive rate at the SisFall threshold.}",
         r"\label{tab:ablation}", r"\begin{tabular}{lccc}", r"\toprule",
         r"Model & Source statistics & Domain statistics & Subject statistics \\", r"\midrule"]
for arch in C.ARCHS:
    cells = [f"{pm(mu[(v, arch)], sd[(v, arch)])} ({100 * rec_mu[(v, arch)]:.1f}\\% / {100 * fpr_mu[(v, arch)]:.1f}\\%)"
             for v in VIEWS]
    lines.append(" & ".join([arch] + cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_ablation.tex", "\n".join(lines))

--- table_ablation.tex
\begin{table}[ht]
\centering
\footnotesize
\caption{Zero-shot Daphnet results under three label-free normalisations of the target inputs: AUROC (mean $\pm$ std over seeds) and, in brackets, recall / false-positive rate at the SisFall threshold.}
\label{tab:ablation}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lccc}
\toprule
Model & Source statistics & Domain statistics & Subject statistics \\
\midrule
CNN-1D & 0.621 $\pm$ 0.064 (0.0\% / 0.2\%) & 0.491 $\pm$ 0.056 (0.9\% / 0.6\%) & 0.528 $\pm$ 0.042 (0.8\% / 2.0\%) \\
LSTM & 0.547 $\pm$ 0.064 (0.0\% / 0.1\%) & 0.587 $\pm$ 0.032 (0.1\% / 0.4\%) & 0.554 $\pm$ 0.031 (0.0\% / 0.4\%) \\
GRU & 0.571 $\pm$ 0.051 (0.0\% / 0.1\%) & 0.631 $\pm$ 0.042 (0.0\% / 0.3\%) & 0.604 $\pm$ 0.027 (0.0\% / 0.3\%) \\
Transformer & 0.603 $\pm$ 0.087 (0.0\% / 0.0\%) & 0.620 $\pm$ 0.036 (0.0\% / 0.5\%) & 0.596 $\pm$ 0.054 (0.0\% / 0.7\%) \\
\bottomrule
\end{tabular}
\end{adjustbox}
\end{table}


## Target-domain references and fine-tuning

In [7]:
ad = pd.read_csv(C.METRICS / "target_adaptation_per_fold.csv")
ORDER = ["zero-shot", "scratch · 2 patients", "SisFall pre-trained · 2 patients",
         "scratch · 6 patients", "SisFall pre-trained · 6 patients"]
NAMES = {"zero-shot": "Zero-shot (no target labels)", "scratch · 2 patients": "Scratch, 2 patients",
         "SisFall pre-trained · 2 patients": "Fine-tuned, 2 patients", "scratch · 6 patients": "Scratch, 6 patients",
         "SisFall pre-trained · 6 patients": "Fine-tuned, 6 patients"}
g = ad.groupby(["condition", "arch"])
lines = [r"\begin{table}[ht]", r"\centering", r"\footnotesize",
         r"\caption{Daphnet AUROC / AUPRC on held-out patients, 5-fold patient-grouped cross-validation "
         r"(mean $\pm$ std over folds). FoG prevalence is 10\%, the AUPRC of a random detector.}",
         r"\label{tab:adaptation}", r"\setlength{\tabcolsep}{3pt}", r"\begin{tabular}{lcccc}", r"\toprule",
         "Setting & " + " & ".join(C.ARCHS) + r" \\", r"\midrule"]
for cond in ORDER:
    cells = []
    for arch in C.ARCHS:
        a_mu, a_sd = g.auroc.mean()[(cond, arch)], g.auroc.std()[(cond, arch)]
        p_mu = g.auprc.mean()[(cond, arch)]
        cells.append(f"{a_mu:.3f}$\\pm${a_sd:.3f} / {p_mu:.3f}")
    lines.append(NAMES[cond] + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_adaptation.tex", "\n".join(lines))

pe = pd.read_csv(C.METRICS / "pretraining_effect.csv")
lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{Effect of SisFall pre-training on Daphnet AUROC: mean difference (fine-tuned $-$ scratch), "
         r"folds improved out of five, and Wilcoxon signed-rank $p$ (five folds, so $p\ge0.0625$).}",
         r"\label{tab:pretrain}", r"\begin{tabular}{llrrrrr}", r"\toprule",
         r"Model & Budget & Scratch & Fine-tuned & $\Delta$ & Improved & $p$ \\", r"\midrule"]
for _, r in pe.iterrows():
    lines.append(f"{r.arch} & {r.budget} & {r.scratch:.3f} & {r['pre-trained']:.3f} & {r['mean diff']:+.3f} & "
                 f"{int(r['folds improved'])}/5 & {r['wilcoxon p']:.3f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_pretrain.tex", "\n".join(lines))

--- table_adaptation.tex
\begin{table}[ht]
\centering
\footnotesize
\caption{Daphnet AUROC / AUPRC on held-out patients, 5-fold patient-grouped cross-validation (mean $\pm$ std over folds). FoG prevalence is 10\%, the AUPRC of a random detector.}
\label{tab:adaptation}
\setlength{\tabcolsep}{3pt}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccc}
\toprule
Setting & CNN-1D & LSTM & GRU & Transformer \\
\midrule
Zero-shot (no target labels) & 0.395$\pm$0.220 / 0.114 & 0.549$\pm$0.235 / 0.123 & 0.641$\pm$0.199 / 0.155 & 0.687$\pm$0.128 / 0.163 \\
Scratch, 2 patients & 0.714$\pm$0.123 / 0.238 & 0.351$\pm$0.256 / 0.104 & 0.548$\pm$0.255 / 0.195 & 0.696$\pm$0.153 / 0.215 \\
Fine-tuned, 2 patients & 0.617$\pm$0.144 / 0.161 & 0.435$\pm$0.170 / 0.089 & 0.588$\pm$0.167 / 0.132 & 0.668$\pm$0.195 / 0.179 \\
Scratch, 6 patients & 0.786$\pm$0.147 / 0.387 & 0.608$\pm$0.251 / 0.202 & 0.769$\pm$0.189 / 0.373 & 0.776$\pm$0.178 / 0.350 \\
Fine-tuned, 6 patients & 0.727$\pm$0.175 / 0.302 & 0.4

## Shift measures

In [8]:
sh = pd.read_csv(C.METRICS / "shift_measures.csv").set_index("space")
order = ["input statistics"] + C.ARCHS
lines = [r"\begin{table}[ht]", r"\centering", r"\footnotesize",
         r"\caption{Shift between SisFall ADL and Daphnet walking, in input space and in the embedding of "
         r"each SisFall-trained model: squared MMD, AUROC of a linear domain probe, under source and "
         r"domain normalisation; and cosine between the fall and FoG class-difference directions.}",
         r"\label{tab:shift}", r"\setlength{\tabcolsep}{4pt}", r"\begin{tabular}{lccccc}", r"\toprule",
         r" & \multicolumn{2}{c}{MMD$^2$} & \multicolumn{2}{c}{Domain probe AUROC} & \\",
         r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
         r"Space & source st. & domain st. & source st. & domain st. & $\cos(\Delta_\mathcal{S},\Delta_\mathcal{T})$ \\",
         r"\midrule"]
for sp in order:
    r = sh.loc[sp]
    lines.append(f"{sp.capitalize() if sp == 'input statistics' else sp} & "
                 f"{r['MMD² ADL vs walk (source stats)']:.3f} & {r['MMD² ADL vs walk (domain stats)']:.3f} & "
                 f"{r['domain probe AUROC (source stats)']:.3f} & {r['domain probe AUROC (domain stats)']:.3f} & "
                 f"{r['cos(fall-ADL, FoG-walk)']:+.3f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
write("table_shift.tex", "\n".join(lines))

--- table_shift.tex
\begin{table}[ht]
\centering
\footnotesize
\caption{Shift between SisFall ADL and Daphnet walking, in input space and in the embedding of each SisFall-trained model: squared MMD, AUROC of a linear domain probe, under source and domain normalisation; and cosine between the fall and FoG class-difference directions.}
\label{tab:shift}
\setlength{\tabcolsep}{4pt}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lccccc}
\toprule
 & \multicolumn{2}{c}{MMD$^2$} & \multicolumn{2}{c}{Domain probe AUROC} & \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}
Space & source st. & domain st. & source st. & domain st. & $\cos(\Delta_\mathcal{S},\Delta_\mathcal{T})$ \\
\midrule
Input statistics & 0.110 & 0.072 & 0.958 & 0.995 & +0.766 \\
CNN-1D & 0.200 & 0.081 & 0.994 & 0.995 & +0.182 \\
LSTM & 0.446 & 0.017 & 0.989 & 0.975 & +0.096 \\
GRU & 0.097 & 0.074 & 0.993 & 0.981 & -0.030 \\
Transformer & 0.128 & 0.024 & 0.976 & 0.962 & +0.122 \\
\bottomrule
\end{tabular}
\end{adjustbox}
\end{t

In [9]:
# a compact JSON of headline numbers, used for the README and the poster
head = {
    "sisfall_auroc": test.groupby("arch").auroc.mean().round(4).to_dict(),
    "sisfall_recall": test.groupby("arch").recall.mean().round(4).to_dict(),
    "sisfall_fa_per_hour": test.groupby("arch").false_alarms_per_hour.mean().round(2).to_dict(),
    "daphnet_auroc_domain": zs_main.groupby("arch").auroc.mean().round(4).to_dict(),
    "daphnet_recall_domain": zs_main.groupby("arch").recall.mean().round(4).to_dict(),
    "daphnet_fpr_domain": zs_main.groupby("arch").fpr.mean().round(4).to_dict(),
    "daphnet_auroc_by_norm": {v: zs[zs.normalisation == v].groupby("arch").auroc.mean().round(4).to_dict() for v in VIEWS},
    "target_auroc": {c: ad[ad.condition == c].groupby("arch").auroc.mean().round(4).to_dict() for c in ORDER},
}
json.dump(head, open(C.METRICS / "headline_numbers.json", "w"), indent=2)
print(json.dumps(head, indent=1))

{
 "sisfall_auroc": {
  "CNN-1D": 0.9875,
  "GRU": 0.9993,
  "LSTM": 0.9994,
  "Transformer": 0.9979
 },
 "sisfall_recall": {
  "CNN-1D": 0.9799,
  "GRU": 0.9773,
  "LSTM": 0.9773,
  "Transformer": 0.9589
 },
 "sisfall_fa_per_hour": {
  "CNN-1D": 1.52,
  "GRU": 0.68,
  "LSTM": 0.76,
  "Transformer": 0.93
 },
 "daphnet_auroc_domain": {
  "CNN-1D": 0.491,
  "GRU": 0.631,
  "LSTM": 0.5874,
  "Transformer": 0.6197
 },
 "daphnet_recall_domain": {
  "CNN-1D": 0.0092,
  "GRU": 0.0,
  "LSTM": 0.0006,
  "Transformer": 0.0002
 },
 "daphnet_fpr_domain": {
  "CNN-1D": 0.0065,
  "GRU": 0.0033,
  "LSTM": 0.0038,
  "Transformer": 0.0047
 },
 "daphnet_auroc_by_norm": {
  "source statistics": {
   "CNN-1D": 0.621,
   "GRU": 0.5712,
   "LSTM": 0.5469,
   "Transformer": 0.6028
  },
  "domain statistics": {
   "CNN-1D": 0.491,
   "GRU": 0.631,
   "LSTM": 0.5874,
   "Transformer": 0.6197
  },
  "subject statistics": {
   "CNN-1D": 0.5277,
   "GRU": 0.6042,
   "LSTM": 0.5542,
   "Transformer": 0.5963
  }
 }